# AgentsVille Trip Planner

A multi-stage AI assistant that plans a vacation to the fictional city of **AgentsVille**.

This notebook has two main agents:

1. **ItineraryAgent** — generates an initial day-by-day itinerary from a structured `VacationInfo` input in a single LLM call.
2. **ItineraryRevisionAgent** — refines that itinerary using a **ReAct (THINK → ACT → OBSERVE)** loop with four tools:
   `calculator_tool`, `get_activities_by_date_tool`, `run_evals_tool`, and `final_answer_tool`.

All structured data is validated via Pydantic models defined in `project_lib.py`.


## 1. Setup

Install dependencies (skip if already installed):

```bash
pip install json-repair==0.47.1 numexpr==2.11.0 openai==1.74.0 pandas==2.3.0 pydantic==2.11.7 python-dotenv==1.1.0
```

Then configure your OpenAI API key. If you are on the Udacity / Vocareum workspace, your key starts with `voc-` and must be routed through the Vocareum base URL.


In [1]:
%pip install "json-repair<0.45" "numexpr<2.11" "openai>=1.74,<2" "pandas>=2,<2.3" "pydantic>=2.11,<3" "python-dotenv>=1,<2"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
from datetime import date, datetime, timedelta
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from json_repair import repair_json
from openai import OpenAI
from pydantic import ValidationError

# Local helpers — Pydantic models, mock data, and tool implementations.
from project_lib import (
    Activity,
    DayPlan,
    Traveler,
    TravelPlan,
    VacationInfo,
    WeatherForecast,
    build_available_tools,
    calculator_tool,
    extract_json_block,
    final_answer_tool,
    get_activities_by_date_tool,
    get_activities_for_range,
    get_all_activity_ids,
    get_weather_forecast_range,
    make_run_evals_tool,
    parse_thought_action,
    print_in_box,
)

load_dotenv()  # picks up OPENAI_API_KEY / OPENAI_BASE_URL from a .env file if present


True

In [3]:
# ********** TODO: configure your OpenAI API key **********
#
# Option A (recommended): set OPENAI_API_KEY (and optionally OPENAI_BASE_URL for Vocareum)
# in your shell or a .env file, then leave the code below as-is.
#
# Option B: hard-code the values directly (do NOT commit secrets).

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "voc-REPLACE_ME")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

# Default model — small & cheap, plenty capable for this exercise.
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
print(f"OpenAI client configured. Model = {MODEL}, base_url = {OPENAI_BASE_URL}")


OpenAI client configured. Model = gpt-4o-mini, base_url = https://openai.vocareum.com/v1


## 2. Define the travelers' vacation

The `VacationInfo` Pydantic model (in `project_lib.py`) captures the destination, dates, travelers, budget, and group interests. Below we build one from a plain JSON dict — exactly as you might receive it from an upstream service or form.


In [4]:
# A sample group: two travelers, a long-weekend break in AgentsVille focused on
# art / tech / food. Dates align with the mock weather and activity data.
vacation_info_dict = {
    "destination": "AgentsVille",
    "date_of_arrival": "2025-06-10",
    "date_of_departure": "2025-06-13",
    "budget": 1200.0,
    "travelers": [
        {"name": "Alex Rivera", "age": 32, "interests": ["technology", "ai", "food"]},
        {"name": "Sam Chen", "age": 29, "interests": ["art", "music", "food"]},
    ],
    "interests": ["technology", "art", "food", "culture"],
}

# ********** TODO: read the dict into a VacationInfo Pydantic instance **********
vacation_info = VacationInfo(**vacation_info_dict)

print_in_box(
    f"Destination : {vacation_info.destination}\n"
    f"Dates       : {vacation_info.date_of_arrival} -> {vacation_info.date_of_departure} "
    f"({vacation_info.num_days} days)\n"
    f"Budget      : ${vacation_info.budget:.0f}\n"
    f"Travelers   : " + ", ".join(t.name for t in vacation_info.travelers) + "\n"
    f"Interests   : " + ", ".join(vacation_info.interests),
    title="Vacation info",
)


+--------------------------------------------------------------------------------------+
| VACATION INFO                                                                        |
+--------------------------------------------------------------------------------------+
| Destination : AgentsVille                                                            |
| Dates       : 2025-06-10 -> 2025-06-13 (4 days)                                      |
| Budget      : $1200                                                                  |
| Travelers   : Alex Rivera, Sam Chen                                                  |
| Interests   : technology, art, food, culture                                         |
+--------------------------------------------------------------------------------------+


## 3. Review weather forecast and available activities

We simulate two bulk "API" calls — one for weather, one for activities — covering the full vacation window. The `start` and `end` dates are read from the `VacationInfo` Pydantic instance above.


In [5]:
# ********** TODO: pull weather + activities for the vacation window **********
weather_data = get_weather_forecast_range(
    vacation_info.date_of_arrival,
    vacation_info.date_of_departure,
)
activities_data = get_activities_for_range(
    vacation_info.date_of_arrival,
    vacation_info.date_of_departure,
)

# Quick visual check
print_in_box(
    "\n".join(
        f"{w['date']}: {w['condition']:14s}  high {w['high_temperature_celsius']:.0f}C  "
        f"low {w['low_temperature_celsius']:.0f}C  precip {int(w['precipitation_chance']*100)}%  "
        f"— {w['description']}"
        for w in weather_data
    ),
    title="Weather forecast",
)
print()
print(f"Available activities in window: {len(activities_data)}")
pd.DataFrame(activities_data)[["activity_id", "name", "start_time", "price", "related_interests", "indoor"]]


+--------------------------------------------------------------------------------------+
| WEATHER FORECAST                                                                     |
+--------------------------------------------------------------------------------------+
| 2025-06-10: sunny           high 28C  low 18C  precip 5%  — Sunny, warm, and dry.    |
| Great day for outdoor activities.                                                    |
| 2025-06-11: partly cloudy   high 26C  low 17C  precip 20%  — Mostly pleasant with    |
| some clouds. Light breeze in the afternoon.                                          |
| 2025-06-12: rainy           high 21C  low 15C  precip 85%  — Heavy rain expected     |
| throughout the day. Outdoor activities not advised.                                  |
| 2025-06-13: sunny           high 30C  low 20C  precip 0%  — Hot and clear.           |
| Excellent for any outdoor plans, bring sunscreen.                                    |
+--------------------

,activity_id,name,start_time,price,related_interests,indoor
0,ART-001,AgentsVille Modern Art Gallery Tour,2025-06-10T10:00:00,25.0,"[art, culture]",True
1,TECH-001,AI & Robotics Meetup,2025-06-10T18:00:00,0.0,"[technology, ai, networking]",True
2,FOOD-001,Street Food Walking Tour,2025-06-10T19:00:00,45.0,"[food, culture]",False
3,PARK-001,Sunrise Hike at Algorithm Peak,2025-06-10T06:30:00,15.0,"[outdoors, hiking, nature]",False
4,MUS-001,History of Computing Museum Visit,2025-06-11T09:30:00,20.0,"[technology, history, culture]",True
5,FOOD-002,Hands-on Pasta Cooking Class,2025-06-11T14:00:00,65.0,"[food, cooking]",True
6,OUT-001,River Cruise & Picnic,2025-06-11T12:30:00,40.0,"[outdoors, relaxation]",False
7,MUSIC-001,Jazz Night at the Cipher Club,2025-06-11T20:00:00,30.0,"[music, culture, nightlife]",True
8,ART-002,Interactive Digital Art Exhibition,2025-06-12T10:00:00,22.0,"[art, technology]",True
9,TECH-002,Build-Your-Own-Agent Workshop,2025-06-12T13:00:00,85.0,"[technology, ai]",True


## 4. The Itinerary Agent

This agent runs in a single LLM call. The system prompt has to:

- Set the LLM's role as an expert AgentsVille travel planner.
- Guide it through a **Chain-of-Thought** planning process.
- Specify that the output must be a JSON object conforming to the `TravelPlan` Pydantic model — and include that schema inline.
- Make use of the **context** we already gathered: `VacationInfo`, weather, and the activity catalog.


In [6]:
# We inject the TravelPlan JSON schema into the system prompt so the LLM has an
# unambiguous output spec to follow.
TRAVEL_PLAN_SCHEMA = json.dumps(TravelPlan.model_json_schema(), indent=2)

# ********** TODO: design ITINERARY_AGENT_SYSTEM_PROMPT **********
ITINERARY_AGENT_SYSTEM_PROMPT = f"""You are **ItineraryAgent**, an expert AgentsVille travel planner. Your job is to design a thoughtful, day-by-day itinerary tailored to a specific group of travelers, using only the activities available in the supplied catalog and respecting the supplied weather forecast and budget.

## Your reasoning process (Chain-of-Thought — think through these steps before writing JSON)

1. **Read the VacationInfo carefully.** Note the destination, the arrival and departure dates (inclusive), the travelers (with their individual interests and ages), the group-level interests, and the total budget in USD.
2. **Skim the weather forecast.** For each day, note the condition and precipitation chance. Days with high precipitation (>= 0.5) should prefer indoor activities; sunny days are good for outdoor activities.
3. **Review the activity catalog.** For each day in the trip window, identify activities that (a) actually exist in the catalog for that date, (b) align with the travelers' interests, and (c) are appropriate for the day's weather.
4. **Build the daily plan.** Aim for a balanced day with morning / afternoon / evening activities where possible. Avoid double-booking overlapping time slots. Try to mix interest categories across the trip so every traveler has something they care about.
5. **Cost the trip.** For each scheduled activity, the cost contribution is `price × number_of_travelers`. Sum across all activities to get `total_cost`. The `total_cost` MUST be less than or equal to the budget.
6. **Sanity-check** before emitting JSON: every `activity_id` you reference must appear verbatim in the supplied catalog; every date you reference must be in the trip window; the day order must be chronological; weather-incompatible activities must be excluded on rainy days.

## Output format

Return **only** a single JSON object (no prose, no markdown fences) that validates against this `TravelPlan` JSON schema:

```
{TRAVEL_PLAN_SCHEMA}
```

Field guidance:
- `city`: must equal the VacationInfo destination.
- `start_date` / `end_date`: must equal date_of_arrival / date_of_departure.
- `travelers`: copy the travelers exactly as provided.
- `days`: one `DayPlan` per date in the trip window, in chronological order.
- `days[*].activities[*]`: copy the matching activity record from the catalog verbatim (same `activity_id`, `name`, `description`, `location`, `start_time`, `end_time`, `price`, `related_interests`, `indoor`).
- `total_cost`: equals the sum of `price × number_of_travelers` across all scheduled activities; never exceed the budget.
- `summary`: 2–3 sentences describing the spirit of the trip.

## Example reasoning (illustrative, do not copy verbatim)

> Day 1 is sunny — I'll schedule the morning hike (PARK-001) for outdoor lovers, then the indoor art gallery tour (ART-001) midday, and the free tech meetup (TECH-001) in the evening. That gives Alex a tech-focused evening and Sam an art-focused midday, and keeps Day 1 cheap at $80 total for two travelers. Day 2 is partly cloudy — fine for the river cruise (OUT-001) at midday, with the pasta class (FOOD-002) as a backup-friendly indoor afternoon, and jazz (MUSIC-001) in the evening. Day 3 is rainy — strictly indoor: digital art (ART-002), build-your-own-agent workshop (TECH-002), tea tasting (FOOD-003). Day 4 is sunny — kayaking (OUT-003) early, AI ethics panel (TECH-003) afternoon, outdoor symphony (MUSIC-002), rooftop dinner (FOOD-004) to close.

Remember: **only** activities that appear in the supplied catalog are valid. If you do not know an activity_id, do not invent one.
"""
print(ITINERARY_AGENT_SYSTEM_PROMPT[:600], "...")


You are **ItineraryAgent**, an expert AgentsVille travel planner. Your job is to design a thoughtful, day-by-day itinerary tailored to a specific group of travelers, using only the activities available in the supplied catalog and respecting the supplied weather forecast and budget.

## Your reasoning process (Chain-of-Thought — think through these steps before writing JSON)

1. **Read the VacationInfo carefully.** Note the destination, the arrival and departure dates (inclusive), the travelers (with their individual interests and ages), the group-level interests, and the total budget in USD.
2 ...


In [7]:
def generate_initial_itinerary(
    vacation_info: VacationInfo,
    weather_data: List[Dict[str, Any]],
    activities_data: List[Dict[str, Any]],
    *,
    model: str = MODEL,
) -> TravelPlan:
    """Call the LLM once to produce a TravelPlan JSON object."""
    user_message = (
        "Plan the trip described below.\n\n"
        "## VacationInfo\n"
        f"{vacation_info.model_dump_json(indent=2)}\n\n"
        "## Weather forecast\n"
        f"{json.dumps(weather_data, indent=2)}\n\n"
        "## Available activities (the ONLY activities you may schedule)\n"
        f"{json.dumps(activities_data, indent=2)}\n"
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": ITINERARY_AGENT_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.3,
        response_format={"type": "json_object"},
    )

    raw = response.choices[0].message.content or ""
    blob = extract_json_block(raw) or raw
    try:
        data = json.loads(blob)
    except json.JSONDecodeError:
        data = json.loads(repair_json(blob))
    return TravelPlan.model_validate(data)


initial_itinerary = generate_initial_itinerary(vacation_info, weather_data, activities_data)

for day in initial_itinerary.days:
    print_in_box(
        "\n".join(
            f"{a.start_time[11:16]}–{a.end_time[11:16]}  {a.name}  (${a.price:.0f}, {a.activity_id})"
            for a in day.activities
        ),
        title=f"{day.date}",
    )
print(f"\nTotal cost: ${initial_itinerary.total_cost:.2f}  /  Budget: ${vacation_info.budget:.0f}")


+--------------------------------------------------------------------------------------+
| 2025-06-10                                                                           |
+--------------------------------------------------------------------------------------+
| 06:30–09:00  Sunrise Hike at Algorithm Peak  ($15, PARK-001)                         |
| 10:00–12:00  AgentsVille Modern Art Gallery Tour  ($25, ART-001)                     |
| 18:00–20:30  AI & Robotics Meetup  ($0, TECH-001)                                    |
| 19:00–21:30  Street Food Walking Tour  ($45, FOOD-001)                               |
+--------------------------------------------------------------------------------------+
+--------------------------------------------------------------------------------------+
| 2025-06-11                                                                           |
+--------------------------------------------------------------------------------------+
| 09:30–12:00  Histor

## 5. Evaluating the itinerary

A planner that *looks* right is not enough — we need automated checks. We'll build several evaluators and compose them into a single `run_evals` function that returns structured results.

The evaluators we want:

1. **City matches** the requested destination.
2. **Dates match** arrival/departure.
3. **Total cost** matches the sum of activity prices × travelers, *and* is within budget.
4. **No hallucinated activities** — every `activity_id` exists in the catalog.
5. **Weather compatibility** — uses an LLM (`ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT`) to decide whether an outdoor activity is safe given that day's forecast.
6. **Traveler feedback satisfied** — uses an LLM to check that user-supplied feedback (e.g. "at least 2 activities per day") is honored.


In [8]:
# ********** TODO: design ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT **********
ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT = """You are **WeatherCompatibilityJudge**, a careful safety evaluator.

## Task
You are given exactly one activity and the weather forecast for the day on which it is scheduled. Decide whether the activity is reasonable to attend under that weather.

Use these rules:
- If `precipitation_chance` is at least 0.5 and the activity is held outdoors (`indoor` is false), the activity is NOT compatible.
- If the weather `condition` is 'stormy' and the activity is outdoors, the activity is NOT compatible.
- If `high_temperature_celsius` is above 35 and the activity is a strenuous outdoor activity (hiking, kayaking, running), the activity is NOT compatible.
- Indoor activities (`indoor` is true) are ALWAYS compatible regardless of weather.
- All other combinations are compatible.

## Output format
Return ONLY a single compact JSON object — no prose, no markdown — with exactly these two fields:

{"compatible": true | false, "reason": "<one short sentence>"}

## Examples

Input:
  activity = {"name": "Sunrise Hike", "indoor": false}
  weather  = {"condition": "rainy", "precipitation_chance": 0.85, "high_temperature_celsius": 21}
Output:
  {"compatible": false, "reason": "Outdoor hike with 85% precipitation chance is unsafe."}

Input:
  activity = {"name": "History of Computing Museum Visit", "indoor": true}
  weather  = {"condition": "rainy", "precipitation_chance": 0.85, "high_temperature_celsius": 21}
Output:
  {"compatible": true, "reason": "Indoor activity is unaffected by rain."}

Input:
  activity = {"name": "River Cruise & Picnic", "indoor": false}
  weather  = {"condition": "partly cloudy", "precipitation_chance": 0.20, "high_temperature_celsius": 26}
Output:
  {"compatible": true, "reason": "Light cloud cover with low precipitation is fine for an outdoor cruise."}

Input:
  activity = {"name": "Kayaking on Lake Lambda", "indoor": false}
  weather  = {"condition": "sunny", "precipitation_chance": 0.0, "high_temperature_celsius": 37}
Output:
  {"compatible": false, "reason": "Strenuous outdoor activity above 35C is a heat-safety risk."}
"""


def activity_and_weather_are_compatible(
    activity: Activity,
    weather: Dict[str, Any],
    *,
    model: str = MODEL,
) -> Dict[str, Any]:
    """LLM-based check that returns {'compatible': bool, 'reason': str}."""
    user_msg = (
        f"activity = {json.dumps(activity.model_dump(mode='json'))}\n"
        f"weather  = {json.dumps(weather)}"
    )
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content or "{}"
    blob = extract_json_block(raw) or raw
    try:
        return json.loads(blob)
    except json.JSONDecodeError:
        return json.loads(repair_json(blob))


In [9]:
# Non-LLM evaluators — these are deterministic checks.

def eval_city_matches(plan: TravelPlan, info: VacationInfo) -> Dict[str, Any]:
    ok = plan.city.strip().lower() == info.destination.strip().lower()
    return {
        "name": "city_matches",
        "passed": ok,
        "message": "OK" if ok else f"plan.city={plan.city!r} != vacation.destination={info.destination!r}",
    }


def eval_dates_match(plan: TravelPlan, info: VacationInfo) -> Dict[str, Any]:
    start_ok = plan.start_date == info.date_of_arrival
    end_ok = plan.end_date == info.date_of_departure
    days_ok = [d.date for d in plan.days] == info.date_range()
    ok = start_ok and end_ok and days_ok
    msg_parts = []
    if not start_ok:
        msg_parts.append(f"start_date {plan.start_date} != {info.date_of_arrival}")
    if not end_ok:
        msg_parts.append(f"end_date {plan.end_date} != {info.date_of_departure}")
    if not days_ok:
        msg_parts.append(
            f"day dates {[str(d.date) for d in plan.days]} != expected {[str(d) for d in info.date_range()]}"
        )
    return {"name": "dates_match", "passed": ok, "message": "OK" if ok else "; ".join(msg_parts)}


def eval_total_cost(plan: TravelPlan, info: VacationInfo) -> Dict[str, Any]:
    n_travelers = len(info.travelers)
    expected = sum(a.price * n_travelers for d in plan.days for a in d.activities)
    matches_sum = abs(plan.total_cost - expected) < 0.01
    within_budget = plan.total_cost <= info.budget + 1e-6
    ok = matches_sum and within_budget
    msg_parts = []
    if not matches_sum:
        msg_parts.append(f"total_cost={plan.total_cost:.2f} != sum(price)*n_travelers={expected:.2f}")
    if not within_budget:
        msg_parts.append(f"total_cost={plan.total_cost:.2f} exceeds budget={info.budget:.2f}")
    return {"name": "total_cost", "passed": ok, "message": "OK" if ok else "; ".join(msg_parts)}


def eval_no_hallucinated_activities(plan: TravelPlan) -> Dict[str, Any]:
    valid_ids = set(get_all_activity_ids())
    bad = [a.activity_id for d in plan.days for a in d.activities if a.activity_id not in valid_ids]
    ok = not bad
    return {
        "name": "no_hallucinated_activities",
        "passed": ok,
        "message": "OK" if ok else f"unknown activity_id(s): {sorted(set(bad))}",
    }


def eval_weather_compatible(plan: TravelPlan, weather_by_date: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    bad: List[str] = []
    for day in plan.days:
        w = weather_by_date.get(str(day.date))
        if not w:
            bad.append(f"no weather for {day.date}")
            continue
        for a in day.activities:
            verdict = activity_and_weather_are_compatible(a, w)
            if not verdict.get("compatible", True):
                bad.append(f"{day.date} '{a.name}' — {verdict.get('reason', '')}")
    ok = not bad
    return {
        "name": "weather_compatible",
        "passed": ok,
        "message": "OK" if ok else "; ".join(bad),
    }


# Compose the weather lookup once for cheap re-use
weather_by_date = {w["date"]: w for w in weather_data}


In [10]:
# Traveler feedback evaluator (LLM-based) — used by the revision agent.

TRAVELER_FEEDBACK_SYSTEM_PROMPT = """You are **TravelerFeedbackJudge**. You decide whether a proposed travel itinerary satisfies a single, specific piece of feedback from the travelers.

## Task
You are given a TravelPlan (JSON) and a single feedback string. Decide whether the plan satisfies that feedback.

## Output
Return ONLY a single compact JSON object — no prose, no markdown — with exactly these two fields:

{"satisfied": true | false, "reason": "<one short sentence>"}

## Examples

feedback = "Make sure there are at least 2 activities per day."
plan = days=[Day1: [A,B], Day2: [C,D,E]]
-> {"satisfied": true, "reason": "Every day has at least 2 activities."}

feedback = "Make sure there are at least 2 activities per day."
plan = days=[Day1: [A], Day2: [C,D]]
-> {"satisfied": false, "reason": "Day 1 has only 1 activity."}

feedback = "Include at least one free activity."
plan = days=[Day1: [A($25), B($0)], Day2: [C($30)]]
-> {"satisfied": true, "reason": "Activity B on Day 1 is free."}
"""


def eval_traveler_feedback(plan: TravelPlan, feedback: str, *, model: str = MODEL) -> Dict[str, Any]:
    if not feedback.strip():
        return {"name": "traveler_feedback", "passed": True, "message": "no feedback supplied"}
    user_msg = (
        f"feedback = {json.dumps(feedback)}\n"
        f"plan = {plan.model_dump_json()}"
    )
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": TRAVELER_FEEDBACK_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.0,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content or "{}"
    blob = extract_json_block(raw) or raw
    try:
        verdict = json.loads(blob)
    except json.JSONDecodeError:
        verdict = json.loads(repair_json(blob))
    ok = bool(verdict.get("satisfied", False))
    return {"name": "traveler_feedback", "passed": ok, "message": verdict.get("reason", "")}


In [11]:
def run_evals(
    plan_dict_or_obj: Any,
    *,
    info: VacationInfo = vacation_info,
    weather_by_date: Dict[str, Dict[str, Any]] = weather_by_date,
    traveler_feedback: str = "",
) -> Dict[str, Any]:
    """Run every evaluator and aggregate results."""
    if isinstance(plan_dict_or_obj, TravelPlan):
        plan = plan_dict_or_obj
    else:
        plan = TravelPlan.model_validate(plan_dict_or_obj)

    results = [
        eval_city_matches(plan, info),
        eval_dates_match(plan, info),
        eval_total_cost(plan, info),
        eval_no_hallucinated_activities(plan),
        eval_weather_compatible(plan, weather_by_date),
        eval_traveler_feedback(plan, traveler_feedback),
    ]
    all_passed = all(r["passed"] for r in results)
    return {"all_passed": all_passed, "results": results}


# Quick run against the initial itinerary
initial_eval = run_evals(initial_itinerary)
print_in_box(
    "\n".join(
        f"[{'PASS' if r['passed'] else 'FAIL'}] {r['name']}: {r['message']}"
        for r in initial_eval["results"]
    ) + f"\n\nALL PASSED: {initial_eval['all_passed']}",
    title="Initial itinerary evaluation",
)


+--------------------------------------------------------------------------------------+
| INITIAL ITINERARY EVALUATION                                                         |
+--------------------------------------------------------------------------------------+
| [PASS] city_matches: OK                                                              |
| [PASS] dates_match: OK                                                               |
| [FAIL] total_cost: total_cost=490.00 != sum(price)*n_travelers=1084.00               |
| [PASS] no_hallucinated_activities: OK                                                |
| [PASS] weather_compatible: OK                                                        |
| [PASS] traveler_feedback: no feedback supplied                                       |
|                                                                                      |
| ALL PASSED: False                                                                    |
+--------------------

## 6. Defining the tools

The four tools the revision agent has access to live in `project_lib.py`:

| Tool | Purpose |
|---|---|
| `calculator_tool` | Evaluate arithmetic expressions accurately (no floating-point hallucinations). |
| `get_activities_by_date_tool` | Fetch the activity catalog for a single date. |
| `run_evals_tool` | Run the evaluator suite against a proposed plan. |
| `final_answer_tool` | Submit the final answer and exit the ReAct loop. |

Because `run_evals_tool` needs access to the closure that includes the OpenAI client, we bind it here via `make_run_evals_tool` and then build the registry.

The docstring of `get_activities_by_date_tool` (defined in `project_lib.py`) describes:

- The tool's purpose (look up bookable activities on a single date).
- The input parameter: `date: str` in **'YYYY-MM-DD'** ISO 8601 form.
- The return shape: a list of activity dicts with stable keys.


In [12]:
traveler_feedback = "Make sure every day has at least 2 activities."

# Bind run_evals_tool so it uses the closure with `client` and `vacation_info`.
bound_run_evals_tool = make_run_evals_tool(
    lambda plan: run_evals(plan, traveler_feedback=traveler_feedback)
)

available_tools = build_available_tools(bound_run_evals_tool)

# Sanity print
for name, spec in available_tools.items():
    print(f"- {name}({', '.join(f'{p}: {t}' for p, t in spec['parameters'].items())})")
    print(f"    {spec['description']}")

# Confirm the get_activities_by_date_tool docstring satisfies the rubric:
print()
print_in_box(get_activities_by_date_tool.__doc__ or "", title="get_activities_by_date_tool docstring")


- calculator_tool(expression: str)
    Evaluate a pure-arithmetic expression (e.g. '25 + 65*2 + 40'). Use this for totaling activity prices or multiplying by number of travelers. Returns the numeric result as a string.
- get_activities_by_date_tool(date: str (YYYY-MM-DD))
    Look up available activities in AgentsVille for a single date. Returns a list of activity records (activity_id, name, description, location, start_time, end_time, price, related_interests, indoor).
- run_evals_tool(travel_plan: dict (TravelPlan JSON))
    Run all evaluation criteria against a proposed travel plan. Returns {'all_passed': bool, 'results': [...]} so you can see which checks fail.
- final_answer_tool(travel_plan: dict (TravelPlan JSON))
    Submit the final, revised TravelPlan. Calling this signals the end of the ReAct loop. Only call this AFTER run_evals_tool reports all checks pass.

+--------------------------------------------------------------------------------------+
| GET_ACTIVITIES_BY_DATE_TOO

## 7. The Itinerary Revision Agent (ReAct)

The revision agent runs a **THINK → ACT → OBSERVE** loop:

- **THINK** — reason about what is currently wrong with the plan (or whether it's already good).
- **ACT** — emit a single tool call as `{"tool_name": "...", "arguments": {...}}`.
- **OBSERVE** — Python executes the tool and feeds the result back as the next user message.

The loop exits when the agent calls `final_answer_tool`. Per the rubric, the agent must call `run_evals_tool` at least once **before** invoking `final_answer_tool`, both to get initial feedback and to confirm the revised plan passes.


In [13]:
def _format_tools_block(tools: Dict[str, Dict[str, Any]]) -> str:
    """Render the available_tools registry into a string the LLM can read."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(f"{p}: {t}" for p, t in spec["parameters"].items())
        lines.append(f"- **{name}**({params})\n    {spec['description']}")
    return "\n".join(lines)


# ********** TODO: design ITINERARY_REVISION_AGENT_SYSTEM_PROMPT **********
ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = f"""You are **ItineraryRevisionAgent**, an expert AgentsVille travel planner whose job is to revise a proposed travel itinerary until it passes every evaluation check, while honoring the travelers' feedback.

## Your task
You will be given:
1. The original `VacationInfo` (destination, dates, travelers, budget, interests).
2. The current proposed `TravelPlan` (JSON).
3. Any feedback from the travelers (free-form string, e.g. "at least 2 activities per day").

Use the available tools to **diagnose** what is wrong with the current plan, **fix** it, **verify** that the fix passes all checks, and then **submit** the final answer via `final_answer_tool`. The final plan you submit MUST validate against the `TravelPlan` Pydantic schema below.

## The ReAct cycle
On every assistant turn, return **exactly one message** with both of these sections, in this order and with these exact headings:

THOUGHT: <your private reasoning — what you observed last turn, what the plan still needs, which tool you'll call next and why>
ACTION: {{"tool_name": "<one of the available tools>", "arguments": {{"<arg1>": <value>, ...}}}}

After your ACTION, Python will execute the tool and reply with an OBSERVATION message containing the tool's return value as JSON. Use the OBSERVATION to inform your next THOUGHT/ACTION pair.

Rules for the ACTION block:
- It MUST be a single, valid JSON object on a single logical line (no trailing prose, no markdown fences, no comments).
- The schema is exactly: {{"tool_name": "[tool_name]", "arguments": {{"arg1": "value1", ...}}}}
- `tool_name` MUST be one of the tools listed below.
- `arguments` MUST contain every required parameter for that tool, with the right types.

## Available tools
{_format_tools_block(available_tools)}

## Required workflow
1. FIRST, call `run_evals_tool` against the current proposed plan to see which checks are failing and why. Do this even if the plan looks good.
2. Based on the failing checks, use `get_activities_by_date_tool` to look up replacement activities on the affected dates, and `calculator_tool` to recompute `total_cost` whenever you change the activity list.
3. When you believe the plan is correct, you MUST call `run_evals_tool` AGAIN on the revised plan to verify that every check now passes. Do NOT skip this re-verification step.
4. ONLY after `run_evals_tool` reports `all_passed: true` should you call `final_answer_tool` with the revised plan. Calling `final_answer_tool` ends the loop, so calling it on a still-failing plan is a hard failure.

## TravelPlan schema (your `final_answer_tool` argument must conform)

```
{TRAVEL_PLAN_SCHEMA}
```

## Format examples

Good first turn:
THOUGHT: I have not seen any eval feedback yet, so I should run evals on the proposed plan to find what to fix.
ACTION: {{"tool_name": "run_evals_tool", "arguments": {{"travel_plan": <the proposed plan as JSON>}}}}

Good fix turn:
THOUGHT: The traveler_feedback check failed because Day 2025-06-12 only has 1 activity. I'll look up what else is bookable that day.
ACTION: {{"tool_name": "get_activities_by_date_tool", "arguments": {{"date": "2025-06-12"}}}}

Good exit turn (only after evals show all_passed=true on the revised plan):
THOUGHT: The latest run_evals_tool call showed all_passed=true. I'll submit the revised plan.
ACTION: {{"tool_name": "final_answer_tool", "arguments": {{"travel_plan": <revised plan as JSON>}}}}
"""
print(ITINERARY_REVISION_AGENT_SYSTEM_PROMPT[:800], "...")


You are **ItineraryRevisionAgent**, an expert AgentsVille travel planner whose job is to revise a proposed travel itinerary until it passes every evaluation check, while honoring the travelers' feedback.

## Your task
You will be given:
1. The original `VacationInfo` (destination, dates, travelers, budget, interests).
2. The current proposed `TravelPlan` (JSON).
3. Any feedback from the travelers (free-form string, e.g. "at least 2 activities per day").

Use the available tools to **diagnose** what is wrong with the current plan, **fix** it, **verify** that the fix passes all checks, and then **submit** the final answer via `final_answer_tool`. The final plan you submit MUST validate against the `TravelPlan` Pydantic schema below.

## The ReAct cycle
On every assistant turn, return **exact ...


In [14]:
MAX_REACT_STEPS = 12


def run_revision_agent(
    initial_plan: TravelPlan,
    vacation_info: VacationInfo,
    traveler_feedback: str,
    *,
    model: str = MODEL,
    max_steps: int = MAX_REACT_STEPS,
    verbose: bool = True,
) -> TravelPlan:
    """Drive the ReAct loop until final_answer_tool is invoked (or max_steps)."""
    messages: List[Dict[str, str]] = [
        {"role": "system", "content": ITINERARY_REVISION_AGENT_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                "## VacationInfo\n"
                f"{vacation_info.model_dump_json(indent=2)}\n\n"
                "## Traveler feedback\n"
                f"{traveler_feedback}\n\n"
                "## Current proposed TravelPlan\n"
                f"{initial_plan.model_dump_json(indent=2)}\n\n"
                "Begin the THINK/ACT/OBSERVE cycle. Your first move should be to evaluate the current plan."
            ),
        },
    ]

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n--- step {step} ---")
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.2,
        )
        assistant_msg = response.choices[0].message.content or ""
        messages.append({"role": "assistant", "content": assistant_msg})

        thought, action = parse_thought_action(assistant_msg)
        if verbose:
            print_in_box(thought or "(no thought parsed)", title=f"THOUGHT (step {step})")
            print_in_box(json.dumps(action, indent=2) if action else "(no action parsed)",
                         title=f"ACTION (step {step})")

        if not action or "tool_name" not in action:
            messages.append({
                "role": "user",
                "content": "OBSERVATION: ERROR — could not parse an ACTION. Please reply with THOUGHT: ... then ACTION: {\"tool_name\": ..., \"arguments\": {...}} on the next turn."
            })
            continue

        tool_name = action["tool_name"]
        arguments = action.get("arguments", {}) or {}

        if tool_name not in available_tools:
            obs = f"ERROR: unknown tool '{tool_name}'. Pick one of: {list(available_tools)}"
            messages.append({"role": "user", "content": f"OBSERVATION: {obs}"})
            continue

        tool_fn = available_tools[tool_name]["function"]
        try:
            result = tool_fn(**arguments)
        except TypeError as exc:
            obs = f"ERROR calling {tool_name}: {exc}. Check the arguments dict."
            messages.append({"role": "user", "content": f"OBSERVATION: {obs}"})
            continue
        except ValidationError as exc:
            obs = f"ERROR: travel_plan did not validate against the TravelPlan schema: {exc.errors()[:3]}..."
            messages.append({"role": "user", "content": f"OBSERVATION: {obs}"})
            continue
        except Exception as exc:  # pragma: no cover
            obs = f"ERROR calling {tool_name}: {exc}"
            messages.append({"role": "user", "content": f"OBSERVATION: {obs}"})
            continue

        if tool_name == "final_answer_tool":
            if verbose:
                print_in_box("final_answer_tool called — exiting loop.", title=f"OBSERVATION (step {step})")
            return TravelPlan.model_validate(result["travel_plan"])

        obs_text = json.dumps(result, indent=2, default=str)
        if verbose:
            print_in_box(obs_text[:1200] + ("..." if len(obs_text) > 1200 else ""),
                         title=f"OBSERVATION (step {step})")
        messages.append({"role": "user", "content": f"OBSERVATION: {obs_text}"})

    raise RuntimeError(f"ReAct loop did not converge within {max_steps} steps.")


revised_itinerary = run_revision_agent(
    initial_itinerary,
    vacation_info,
    traveler_feedback=traveler_feedback,
)

print()
final_eval = run_evals(revised_itinerary, traveler_feedback=traveler_feedback)
print_in_box(
    "\n".join(
        f"[{'PASS' if r['passed'] else 'FAIL'}] {r['name']}: {r['message']}"
        for r in final_eval["results"]
    ) + f"\n\nALL PASSED: {final_eval['all_passed']}",
    title="Final itinerary evaluation",
)



--- step 1 ---
+--------------------------------------------------------------------------------------+
| THOUGHT (STEP 1)                                                                     |
+--------------------------------------------------------------------------------------+
| I need to evaluate the current travel plan to see if it meets all the requirements,  |
| especially the traveler feedback about having at least 2 activities per day. I'll    |
| run the evaluation tool to check for any issues.                                     |
+--------------------------------------------------------------------------------------+
+--------------------------------------------------------------------------------------+
| ACTION (STEP 1)                                                                      |
+--------------------------------------------------------------------------------------+
| {                                                                                    |
|   "

## 8. A narrative summary — just for fun

A short marketing-style write-up of the finalized trip, so the travelers actually want to go.


In [15]:
NARRATIVE_SYSTEM_PROMPT = """You are a warm, lyrical travel writer. Given a finalized TravelPlan JSON, write a short, vivid narrative (3-5 short paragraphs) that walks the travelers through their upcoming trip day by day. Mention the travelers by first name, call out 1-2 highlight activities per day, and end on an inviting closing line. Keep it under 350 words. Do not output JSON."""


def write_trip_narrative(plan: TravelPlan, *, model: str = MODEL) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": NARRATIVE_SYSTEM_PROMPT},
            {"role": "user", "content": plan.model_dump_json(indent=2)},
        ],
        temperature=0.7,
    )
    return response.choices[0].message.content or ""


narrative = write_trip_narrative(revised_itinerary)
print_in_box(narrative, title="Your AgentsVille adventure")


+--------------------------------------------------------------------------------------+
| YOUR AGENTSVILLE ADVENTURE                                                           |
+--------------------------------------------------------------------------------------+
| As the sun rises on June 10, Alex and Sam embark on their adventure in AgentsVille   |
| with a refreshing sunrise hike at Algorithm Peak. The cool morning air fills their   |
| lungs as they take in the panoramic views that stretch endlessly before them.        |
| Later, they trade mountain trails for the vibrant streets of the Downtown Art        |
| District, where they enjoy a guided tour of the renowned modern art gallery,         |
| marveling at AI-inspired sculptures that spark their creativity. As evening falls,   |
| the duo dives into the bustling Night Market District for a street food walking      |
| tour, sampling an array of local delicacies that tantalize their taste buds.         |
|                    

## Wrap up

This notebook demonstrated:

1. Structured input via Pydantic (`VacationInfo`).
2. A single-shot LLM call that produces a `TravelPlan` JSON matching the model schema, guided by a Chain-of-Thought system prompt.
3. A composite evaluator suite — five deterministic checks plus an LLM-based weather-compatibility judge and an LLM-based traveler-feedback judge.
4. A ReAct revision agent that reasons in `THOUGHT`/`ACTION` turns, calls four tools, and exits via `final_answer_tool` only after `run_evals_tool` confirms the plan passes.

Things you can try next:
- Change `traveler_feedback` to add a new constraint (e.g. "at least one free activity per day") and watch the revision agent adapt.
- Add a new tool — e.g. `find_restaurants_tool` — to `project_lib.py` and add it to the registry; the prompt picks it up automatically because the tool list is rendered dynamically.
- Swap the model (e.g. to `gpt-4o`) for more sophisticated reasoning.
